# Figure 1D — REDUX overview (GIAB HG008)

Four-panel figure summarising REDUX's read-processing steps — unmapping (A),
deduplication/consensus (B), base-quality recalibration (C) and microsatellite jitter (D) —
from the REDUX output of a single oncoanalyser run.

The published panel uses the GIAB **HG008** Illumina WGS tumour/normal run. Point the
Configuration cell at any other run folder — or at your own oncoanalyser output — and run all
cells; see **Data access** below for what is available and how to get it.

## Data access

Every input below is public **oncoanalyser** output, published in two Google Cloud Storage
buckets. Nothing else is needed to reproduce this figure.

| Bucket | Content | Size |
| --- | --- | --- |
| `gs://hartwig-oapaper-celllines` | 5 cell lines x 3 sequencing platforms, WGS and targeted panel | 36 GB |
| `gs://hartwig-oapaper-patients` | 2 consented patients, fresh-frozen and FFPE WGS, targeted panel | — |

### Folder scheme

Every pipeline run has one folder, built from the same fields in the same order —
**sample, modality, platform, aligner, run type, pipeline**:

```text
gs://hartwig-oapaper-celllines/<sample>/<modality>/<platform>/<aligner>/<run type>/<pipeline>/
gs://hartwig-oapaper-patients/<sample>/<modality>/<preservation>/<platform>/<aligner>/<run type>/<pipeline>/

<sample>        COLO829 | HCC1187 | HCC1395 | HCC1954 | HG008      cell lines
                H00005238 | H00006429                              patients
<modality>      wgs | panel                 panel = HMF oncopanel v3, Illumina only
<preservation>  ff | ffpe                   patients only
<platform>      illumina_novaseq6000 | roche_sbx | ultima_ug100
<aligner>       bwa
<run type>      tn = tumour/normal | to = tumour-only
<pipeline>      oa3 = oncoanalyser 3.x
```

So COLO829, Illumina WGS, tumour/normal is:

```text
gs://hartwig-oapaper-celllines/COLO829/wgs/illumina_novaseq6000/bwa/tn/oa3/
```

Inside `oa3/` are the oncoanalyser output directories exactly as the pipeline writes them —
`alignments/`, `bamtools/`, `purple/`, `sage/`, `linx/`, `sigs/`, `logs/`, ... — with **no**
`results/<subject>` level in between.

Sample names inside a run are *not* uniform: tumours are `<sample>T`, but normals are
`<sample>R` for COLO829, HG008 and the patients and `<sample>BL` for HCC1187/HCC1395/HCC1954,
and the Illumina COLO829 WGS run uses `COLO829v012T` / `COLO829v012R` throughout. Nothing
hard-codes them — every notebook reads the real names out of `alignments/` and `purple/`.

Browse it with:

```bash
gsutil ls gs://hartwig-oapaper-celllines/
gsutil ls gs://hartwig-oapaper-celllines/COLO829/wgs/illumina_novaseq6000/bwa/tn/oa3/
```

### What this notebook reads (~11 MB)

One run folder: **HG008**, Illumina WGS, tumour/normal.

```text
gs://hartwig-oapaper-celllines/HG008/wgs/illumina_novaseq6000/bwa/tn/oa3/
  logs/HG008_HG008T.redux.command.log          panel A   unmapping counts
  logs/HG008_HG008R.redux.command.log          panel A
  alignments/HG008T/HG008T.redux.duplicate_freq.tsv    panel B   duplicate-group sizes
  alignments/HG008T/HG008T.redux.bqr.tsv               panel C   base-quality recalibration
  alignments/HG008T/HG008T.redux.ms_table.tsv.gz       panel D   microsatellite jitter
  alignments/HG008T/HG008T.redux.jitter_params.tsv     panel D
  alignments/HG008R/...                                the same four files for the normal
```

### Option 1 — read straight from the bucket (the default; nothing to download first)

Leave the `gs://` values in the Configuration cell as they are and run all cells. Only the
files listed above are staged, into `data/`, and re-used on every later run — delete that
directory to re-fetch. You need the
[Google Cloud SDK](https://cloud.google.com/sdk/docs/install) on `PATH`, authenticated once:

```bash
gcloud auth login      # any Google account works; the buckets are public, this only sets the billing/quota project
```

### Option 2 — copy the data down first, then work offline

Mirror only the run folders this notebook needs (sizes above), then repoint the
Configuration cell at the copy:

```bash
DEST=~/oapaper-data          # anywhere with enough free space
gcloud storage rsync -r \
  gs://hartwig-oapaper-celllines/HG008/wgs/illumina_novaseq6000/bwa/tn/oa3 \
  $DEST/HG008/wgs/illumina_novaseq6000/bwa/tn/oa3          # 430 MB (whole run)
```

Then, in the Configuration cell:

```python
CELLLINES_ROOT = "/absolute/path/to/oapaper-data"   # the $DEST above
```

Local paths must be **absolute** — `~` is not expanded, so `~/oapaper-data` fails with
`FileNotFoundError`. `gcloud storage rsync` is resumable: re-run it after an interruption.

### Option 3 — run it on your own oncoanalyser output

Point the Configuration cell at any oncoanalyser 3.x run directory, local or `gs://`. The
only requirement is that the directory contains the tool sub-folders listed above; the
sample names are read from the files themselves, so they do not have to match ours.

Running Option 1 once already leaves a portable offline copy of just this figure's inputs in
`data/figure_1/` (~11 MB) — the notebook re-uses it and never re-downloads.

## 1 · Configuration

In [ ]:
from pathlib import Path

# ============================================================================
# CONFIGURATION — edit these to point at your own REDUX output.
#
# RUN_DIR is ONE oncoanalyser run folder (local path or gs:// URI) holding
# alignments/ and logs/. The scheme is
#   <root>/<sample>/<modality>/<platform>/<aligner>/<run type>/<pipeline>
# so switching platform means swapping illumina_novaseq6000 for roche_sbx or
# ultima_ug100, and switching sample means swapping HG008 for COLO829,
# HCC1187, HCC1395 or HCC1954. See the Data access section above.
# ============================================================================
CELLLINES_ROOT = "gs://hartwig-oapaper-celllines"   # or an absolute path to a local copy
SAMPLE_ID      = "HG008"
PLATFORM       = "Illumina"                         # free-text label, used in titles only

RUN_DIR = f"{CELLLINES_ROOT}/{SAMPLE_ID}/wgs/illumina_novaseq6000/bwa/tn/oa3"
# ============================================================================

# Downloaded inputs are cached under data/figure_1/<run-folder>/, keyed on RUN_DIR, so
# switching runs re-fetches rather than reusing the previous run's files. Delete to refresh.
CACHE_ROOT = Path("data") / "figure_1"
EXPORT_DIR = Path("figures"); EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"{SAMPLE_ID}  ({PLATFORM})")
print(f"run: {RUN_DIR}")

## 2 · Locating pipeline output

Reads the tumour and normal sample names out of the run folder configured above, and provides
the local / `gs://` read helpers used by the loading section. Nothing is downloaded here.

In [ ]:
import subprocess, re, hashlib
from pathlib import Path

# ---- IO: identical behaviour for local paths and gs:// URIs -----------------------------------
# Every gsutil call goes through _gsutil, which returns None when the target simply does not exist
# and raises on anything else (auth, permissions, typo'd bucket). A silent [] on a failed listing
# is what makes a missing-data problem look like an empty-run problem.
_MISSING = ("matched no objects", "no urls matched", "404")

def _is_remote(p):
    return str(p).startswith("gs://")

def _gsutil(*args):
    r = subprocess.run(["gsutil", *map(str, args)], capture_output=True, text=True)
    if r.returncode != 0:
        err = r.stderr.strip()
        if any(m in err.lower() for m in _MISSING):
            return None
        raise RuntimeError(f"gsutil {' '.join(map(str, args))} failed:\n{err}")
    return r.stdout

_GLOB_CACHE = {}   # listings are memoised for the session; clear it if a run folder changes

def _glob(folder, pattern, dirs_only=True):
    # Entries matching folder/pattern, without a trailing slash. '*' never crosses a '/'.
    key = (str(folder).rstrip("/"), pattern, dirs_only)
    if key not in _GLOB_CACHE:
        _GLOB_CACHE[key] = _glob_uncached(*key)
    return _GLOB_CACHE[key]

def _glob_uncached(folder, pattern, dirs_only):
    if _is_remote(folder):
        out = _gsutil("ls", "-d", f"{str(folder).rstrip('/')}/{pattern}")
        hits = [l for l in (out or "").splitlines() if l.strip()]
        if dirs_only:
            hits = [h for h in hits if h.endswith("/")]
        return sorted(h.rstrip("/") for h in hits)
    return sorted(str(p) for p in Path(folder).glob(pattern) if p.is_dir() or not dirs_only)

def _cat(src):
    # Read one file as text without caching it (used for the REDUX log, of which only a few
    # lines are kept). Raises when the source is absent.
    if _is_remote(src):
        out = _gsutil("cat", str(src))
        if out is None:
            raise FileNotFoundError(src)
        return out
    return Path(src).read_text()

def _fetch(src, dest):
    # Copy one file into the local cache. Returns None (not an exception) when the source is
    # absent, so a run that skipped a step degrades to a blank panel instead of a crash.
    dest = Path(dest)
    if dest.exists():
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    if _is_remote(src):
        if _gsutil("cp", str(src), str(dest)) is None:
            return None
    else:
        if not Path(src).exists():
            return None
        import shutil; shutil.copy(Path(src), dest)
    return dest

def cache_dir(root, folder, *parts):
    """Local cache path, keyed on the source folder.

    Two runs of the same arm live in different directories, so re-pointing a folder never
    silently reuses the previous run's downloaded files.
    """
    src = str(folder).rstrip("/")
    tail = [p for p in src.split("//")[-1].split("/") if p][-6:]   # readable, still unambiguous
    label = re.sub(r"[^A-Za-z0-9]+", "-", "-".join(tail)).strip("-")
    return Path(root, f"{label}-{hashlib.sha1(src.encode()).hexdigest()[:6]}", *parts)

# ---- Sample discovery -------------------------------------------------------------------------
# The run folder *is* the subject directory, and the sample names come out of alignments/ rather
# than from a naming convention -- this run's normal is HG008R, but HCC1187/HCC1395/HCC1954 use
# <sample>BL and the Illumina COLO829 run uses COLO829v012T / COLO829v012R.
TUMOR_SUFFIXES  = ("T",)
NORMAL_SUFFIXES = ("R", "BL")

def _classify(sample):
    # Tumour vs normal from the sample-name suffix (BL checked before R: "...BL" also ends in 'L').
    for suf in NORMAL_SUFFIXES:
        if sample.endswith(suf):
            return "N"
    for suf in TUMOR_SUFFIXES:
        if sample.endswith(suf):
            return "T"
    return None

def find_samples(base):
    """{"T": sample|None, "N": sample|None} for one run folder, read from <base>/alignments/."""
    found = {"T": None, "N": None}
    for hit in _glob(base, "alignments/*"):
        sample = hit.rsplit("/", 1)[-1]
        tis = _classify(sample)
        if tis and found[tis] is None:
            found[tis] = sample
    return found

SAMPLES = find_samples(RUN_DIR)
TUMOR_NAME, REF_NAME = SAMPLES["T"], SAMPLES["N"]
CACHE_DIR = cache_dir(CACHE_ROOT, RUN_DIR)
if TUMOR_NAME is None:
    raise FileNotFoundError(f"No tumour sample found under {RUN_DIR}/alignments/")

print(f"tumour={TUMOR_NAME}  normal={REF_NAME}")
print(f"cache: {CACHE_DIR}")
print("path helpers ready")

## 3 · Plotting style

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8,
    "axes.titleweight": "bold", "axes.labelweight": "normal",
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.8, "lines.linewidth": 1.4,
    "legend.frameon": False, "svg.fonttype": "none",
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

# COSMIC 6-class single-base-substitution palette
SBS_COLORS = {
    "C>A": "#03BCEE", "C>G": "#000000", "C>T": "#E32926",
    "T>A": "#CACACA", "T>C": "#A1CE63", "T>G": "#EDBFC2",
}
SBS_ORDER = list(SBS_COLORS)

REDUX_BLUE = "#2C6FBB"; ACCENT2 = "#D1495B"; GOOD = "#4C9F70"; WARN = "#E08214"
DUP_CORAL = "#E4572E"
UNMAP_COLORS = {"high depth": ACCENT2, "soft-clip": WARN, "chimeric": REDUX_BLUE}

def human(n):
    n = float(n)
    for unit, div in (("B", 1e9), ("M", 1e6), ("k", 1e3)):
        if abs(n) >= div:
            return f"{n/div:.2f}{unit}"
    return f"{n:.0f}"

## 4 · Load REDUX outputs

Fetches the four small REDUX TSVs for the tumour sample into the cache and parses the REDUX run
log for the unmapping and deduplication counts panels A and B need.

In [ ]:
import pandas as pd

REDUX_FILES = {
    "duplicate_freq": "redux.duplicate_freq.tsv",
    "bqr":            "redux.bqr.tsv",
    "ms_table":       "redux.ms_table.tsv.gz",
    "jitter_params":  "redux.jitter_params.tsv",
}

def redux_log(sample):
    # logs/<subject>_<sample>.redux.command.log -- globbed on the sample, so the subject prefix
    # does not have to match SAMPLE_ID. None when the run published no REDUX log.
    hits = _glob(RUN_DIR, f"logs/*_{sample}.redux.command.log", dirs_only=False)
    return hits[0] if hits else None

def fetch_log_stats(sample):
    # Cache only the handful of relevant REDUX log lines, then parse them. {} if there is no log.
    cache = CACHE_DIR / sample / f"{sample}.redux.logstats.txt"
    if not cache.exists():
        src = redux_log(sample)
        if src is None:
            print(f"WARN: no REDUX log for {sample} under {RUN_DIR}/logs/ -- panels A and B will be bare")
            return {}
        try:
            txt = _cat(src)
        except Exception as e:
            print(f"WARN: could not read REDUX log for {sample}: {e}")
            return {}
        keep = [l for l in txt.splitlines()
                if "unmapped stats:" in l or "stats: totalReads" in l or "unmapping regions from" in l]
        cache.parent.mkdir(parents=True, exist_ok=True)
        cache.write_text("\n".join(keep))
    text = cache.read_text()
    s = {}
    m = re.search(r"reasons\(depth=(\d+) softClip=(\d+) chimeric=(\d+)\)", text)
    if m: s["unmap_reasons"] = {"high depth": int(m[1]), "soft-clip": int(m[2]), "chimeric": int(m[3])}
    m = re.search(r"loaded (\d+) unmapping regions", text)
    if m: s["unmap_regions"] = int(m[1])
    m = re.search(r"totalReads\((\d+)\) duplicates\((\d+)\) dupGroups\((\d+)\) umiGroups\((\d+)\)", text)
    if m: s.update(total_reads=int(m[1]), duplicates=int(m[2]), dup_groups=int(m[3]), umi_groups=int(m[4]))
    return s

def load_redux(sample):
    paths = {}
    for key, fname in REDUX_FILES.items():
        src = f"{RUN_DIR}/alignments/{sample}/{sample}.{fname}"
        got = _fetch(src, CACHE_DIR / sample / f"{sample}.{fname}")
        if got is None:
            raise FileNotFoundError(f"missing REDUX output: {src}")
        paths[key] = got
    return {
        "duplicate_freq": pd.read_csv(paths["duplicate_freq"], sep="\t"),
        "bqr":            pd.read_csv(paths["bqr"], sep="\t"),
        "ms_table":       pd.read_csv(paths["ms_table"], sep="\t", compression="gzip"),
        "jitter_params":  pd.read_csv(paths["jitter_params"], sep="\t"),
        "stats":          fetch_log_stats(sample),
    }

D = load_redux(TUMOR_NAME)
print("Loaded:", {k: v.shape for k, v in D.items() if hasattr(v, "shape")})
D["stats"]

## 5 · Panel A — Unmap problematic regions

In [ ]:
def draw_panel_a(ax, stats):
    ax.set_title("A · Unmap problematic regions", loc="left")
    reasons = stats.get("unmap_reasons")
    if not reasons:
        ax.axis("off"); ax.text(0.5, 0.5, "no log stats", ha="center"); return
    labels = list(reasons); vals = [reasons[k] for k in labels]
    colors = [UNMAP_COLORS[k] for k in labels]
    wedges, _, autot = ax.pie(
        vals, colors=colors, startangle=90, counterclock=False, radius=1.0,
        autopct=lambda p: f"{p:.0f}%" if p > 3 else "", pctdistance=0.68,
        wedgeprops=dict(edgecolor="white", lw=0.9),
        textprops=dict(fontsize=7, color="white", weight="bold"))
    ax.set_aspect("equal")
    ax.legend(wedges, labels, title="unmapped because…", loc="upper center",
              bbox_to_anchor=(0.5, -0.02), ncol=3, fontsize=6.3, title_fontsize=6.5,
              handlelength=1.0, columnspacing=1.0, handletextpad=0.4)
    total = sum(vals)
    bits = [f"≈{human(total)} reads unmapped"]
    if stats.get("total_reads"):
        bits.append(f"~{total/stats['total_reads']:.1%} of reads")
    if stats.get("unmap_regions"):
        bits.append(f"{stats['unmap_regions']:,} regions")
    ax.text(0.5, -0.20, "  ·  ".join(bits), transform=ax.transAxes, ha="center", va="top",
            fontsize=6.6, color="#333")

fig, ax = plt.subplots(figsize=(3.4, 3.0)); draw_panel_a(ax, D["stats"]); plt.show()

## 6 · Panel B — Deduplicate & form consensus reads

In [ ]:
from matplotlib.patches import Rectangle, FancyArrowPatch
import numpy as np

def _base_row(ax, x0, y0, bases, cell=0.46, hi=None, hi_fc=ACCENT2):
    for i, b in enumerate(bases):
        on = (hi is not None and i == hi)
        ax.add_patch(Rectangle((x0 + i*cell, y0), cell*0.9, cell*0.9,
                               fc=(hi_fc if on else "white"), ec="#999", lw=0.5))
        ax.text(x0 + i*cell + cell*0.45, y0 + cell*0.45, b, ha="center", va="center",
                fontsize=5.6, weight="bold", color=("white" if on else "#333"))

def dup_group_fractions(dup_freq):
    # Fraction of read groups by group size, binned as none/2/3/4/>=5.
    d = dup_freq.set_index("DuplicateReadCount")["Frequency"]
    total = d.sum()
    def frac(mask):
        return float(d[mask].sum()) / total
    labels = ["none", "2", "3", "4", "≥5"]
    fracs  = [frac(d.index == 1), frac(d.index == 2), frac(d.index == 3),
              frac(d.index == 4), frac(d.index >= 5)]
    return labels, fracs

def draw_panel_b(ax, stats, dup_freq):
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
    ax.set_title("B · Deduplicate + consensus", loc="left")

    if stats.get("total_reads"):
        dup_pct = stats["duplicates"] / stats["total_reads"]
        ax.text(0.2, 9.4, f"{dup_pct:.1%} duplicates · {human(stats['dup_groups'])} groups collapsed",
                fontsize=7.2, weight="bold", color="#333")

    reads = ["ACGTACG", "ACGTACG", "ACGAACG", "ACGTACG"]   # read 3 has an error at index 3
    for j, seq in enumerate(reads):
        _base_row(ax, 0.5, 8.0 - j*0.60, seq, hi=(3 if j == 2 else None))
    ax.text(0.5, 8.75, "duplicate group", fontsize=6.2, color="#555")
    ax.text(0.5, 5.5, "one read has a sequencing error", fontsize=5.6, color=ACCENT2, ha="left")
    ax.add_patch(FancyArrowPatch((4.05, 7.3), (5.35, 7.3), arrowstyle="-|>", mutation_scale=12,
                                 lw=1.2, color="#555"))
    ax.text(4.7, 7.75, "consensus", fontsize=6.2, color="#333", ha="center", weight="bold")
    _base_row(ax, 5.7, 7.05, "ACGTACG", hi=3, hi_fc=GOOD)
    ax.text(5.7, 6.55, "error corrected, qual recomputed", fontsize=5.2, color=GOOD)

    # Duplicate-family sizes. The "none" bar (~80%) dwarfs the rest, so the y-axis
    # is zoomed to the tail and the "none" bar is truncated with a break mark.
    import matplotlib.ticker as mtick
    ins = ax.inset_axes([0.13, 0.09, 0.83, 0.36])
    labels, fracs = dup_group_fractions(dup_freq)
    x = np.arange(len(labels)); w = 0.68
    YMAX = 0.19
    ins.bar(x, fracs, w, color=DUP_CORAL, edgecolor="white", linewidth=0.4,
            zorder=3, clip_on=True)

    for xi, v in zip(x, fracs):
        if v <= YMAX:
            ins.annotate(f"{v:.1%}", (xi, v), textcoords="offset points", xytext=(0, 1.5),
                         ha="center", va="bottom", fontsize=4.8, color="#333", zorder=6)

    # break mark straddling the top of the truncated "none" bar
    gy, gh, over = 0.163, 0.011, 0.06
    ins.add_patch(Rectangle((-w/2, gy), w, gh, fc="white", ec="none", zorder=5))
    for yy in (gy, gy + gh):
        ins.plot([-w/2 - over, w/2 + over], [yy - 0.006, yy + 0.006],
                 color="#333", lw=0.7, solid_capstyle="round", zorder=6, clip_on=False)
    ins.annotate(f"{fracs[0]:.0%}", (0, YMAX), textcoords="offset points", xytext=(0, 1.5),
                 ha="center", va="bottom", fontsize=5.2, weight="bold", color=DUP_CORAL, zorder=6)

    ins.set_xlim(-0.6, len(labels) - 0.4); ins.set_ylim(0, YMAX)
    ins.set_xticks(x); ins.set_xticklabels(labels)
    ins.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    ins.set_xlabel("duplicates in consensus group", fontsize=6, labelpad=1)
    ins.set_ylabel("read groups", fontsize=6, labelpad=1)
    ins.set_title("duplicate frequency", fontsize=6.2, pad=2)
    ins.grid(axis="y", lw=0.4, alpha=0.35)
    ins.set_axisbelow(True)
    ins.tick_params(labelsize=5.5)
    for sp in ("top", "right"): ins.spines[sp].set_visible(False)

fig, ax = plt.subplots(figsize=(3.4, 3.0)); draw_panel_b(ax, D["stats"], D["duplicate_freq"]); plt.show()

## 7 · Panel C — Base quality recalibration

In [ ]:
import numpy as np

_COMP = {"A": "T", "T": "A", "C": "G", "G": "C"}
def sbs_class(ref, alt):
    return f"{ref}>{alt}" if ref in "CT" else f"{_COMP[ref]}>{_COMP[alt]}"

def bqr_summary(df, read_type="NONE"):
    b = df[(df["ReadType"] == read_type) & (df["Alt"] != df["Ref"])].copy()
    b["sbs"] = [sbs_class(r, a) for r, a in zip(b["Ref"], b["Alt"])]
    g = (b.groupby(["sbs", "OriginalQual"])
           .apply(lambda x: np.average(x["RecalibratedQual"], weights=x["Count"]), include_groups=False)
           .rename("recal").reset_index())
    g["delta"] = g["recal"] - g["OriginalQual"]
    return g

def draw_panel_c(ax, df, *, read_type="NONE"):
    g = bqr_summary(df, read_type)
    quals = sorted(g["OriginalQual"].unique())
    x = np.arange(len(quals)); n = len(SBS_ORDER); w = 0.84 / n
    for i, s in enumerate(SBS_ORDER):
        vals = [g.loc[(g["sbs"] == s) & (g["OriginalQual"] == q), "delta"].squeeze()
                if not g[(g["sbs"] == s) & (g["OriginalQual"] == q)].empty else 0.0 for q in quals]
        ax.bar(x + (i - (n - 1) / 2) * w, vals, w, color=SBS_COLORS[s], label=s,
               edgecolor="white", linewidth=0.3, zorder=3)
    ax.axhline(0, color="black", lw=0.8, zorder=2)
    ax.set_xticks(x); ax.set_xticklabels(quals)
    ax.set_xlabel("Original base quality"); ax.set_ylabel("Quality adjustment ΔQ (phred)")
    ax.set_title("C · Base quality recalibration", loc="left")
    ax.margins(y=0.18)
    ax.text(0.02, 0.97, "ΔQ > 0 : quality raised", transform=ax.transAxes,
            fontsize=6.2, color="#777", va="top")
    ax.legend(title="substitution", ncol=2, loc="upper right", fontsize=6, title_fontsize=6.5,
              handlelength=1.1, columnspacing=0.8, handletextpad=0.4, borderaxespad=0.3)
    return g

fig, ax = plt.subplots(figsize=(3.4, 2.8)); draw_panel_c(ax, D["bqr"]); plt.show()

## 8 · Panel D — Microsatellite jitter

In [ ]:
def _dev_cols(df):
    cols = [c for c in df.columns if c.startswith("count")]
    devs = np.array([int(c.replace("count", "").replace("+", "")) for c in cols])
    o = np.argsort(devs)
    return [cols[i] for i in o], devs[o]

# repeat-class facets
FACETS = {
    "A/T":   ["A/T"],
    "C/G":   ["C/G"],
    "2 bp":  ["AT/TA", "AC/CA/GT/TG", "AG/GA/CT/TC", "CG/GC"],
    "≥3 bp": ["3bp repeat", "4bp repeat", "5bp repeat"],
}
JIT_INS = "#3B6FB6"; JIT_DEL = "#C0392B"

def jitter_phred(ms, units, *, consensus="NONE", min_reads=20000, max_len=20):
    # Per repeat length: insertion / deletion error rate as a Phred score (-10 log10 rate).
    cols, devs = _dev_cols(ms)
    sub = ms[(ms["unit"].isin(units)) & (ms["consensusType"] == consensus)]
    rows = []
    for L, grp in sub.groupby("numUnits"):
        if L > max_len:
            continue
        c = grp[cols].to_numpy().sum(axis=0).astype(float); tot = c.sum()
        if tot < min_reads:
            continue
        ins = c[devs > 0].sum() / tot; dele = c[devs < 0].sum() / tot
        rows.append((int(L),
                     (-10*np.log10(ins) if ins > 0 else np.nan),
                     (-10*np.log10(dele) if dele > 0 else np.nan)))
    return pd.DataFrame(rows, columns=["length", "phred_ins", "phred_del"]).sort_values("length")

def draw_panel_d(fig, spec, ms):
    sub = spec.subgridspec(1, len(FACETS), wspace=0.16)
    axes = []
    for i, (name, units) in enumerate(FACETS.items()):
        a = fig.add_subplot(sub[0, i], sharey=(axes[0] if axes else None))
        d = jitter_phred(ms, units)
        a.plot(d["length"], d["phred_del"], "o", ms=2.8, color=JIT_DEL, label="deletion")
        a.plot(d["length"], d["phred_ins"], "o", ms=2.8, color=JIT_INS, label="insertion")
        a.text(0.5, 0.96, name, transform=a.transAxes, ha="center", va="top",
               fontsize=6.4, weight="bold", color="#444")
        a.set_xlim(3, 21); a.set_xticks([4, 8, 12, 16, 20])
        a.grid(True, lw=0.3, alpha=0.3); a.tick_params(labelsize=6)
        if i > 0:
            a.tick_params(labelleft=False)
        axes.append(a)
    axes[0].set_ylim(8, 54)
    axes[0].set_ylabel("Phred score")
    axes[0].set_title("D · Microsatellite jitter", loc="left")
    axes[0].legend(loc="lower left", fontsize=5.4, handletextpad=0.2, borderaxespad=0.2, labelspacing=0.2)
    fig.canvas.draw()
    p0 = axes[0].get_position(); pN = axes[-1].get_position()
    fig.text((p0.x0 + pN.x1) / 2, p0.y0 - 0.055, "Repeat units", ha="center", fontsize=8)
    return axes

figp = plt.figure(figsize=(6.2, 2.4))
draw_panel_d(figp, figp.add_gridspec(1, 1)[0], D["ms_table"]); plt.show()

## 9 · Composite figure

In [ ]:
fig = plt.figure(figsize=(7.4, 6.2))
gs = fig.add_gridspec(2, 2, hspace=0.40, wspace=0.26,
                      left=0.085, right=0.975, top=0.94, bottom=0.085)
draw_panel_a(fig.add_subplot(gs[0, 0]), D["stats"])
draw_panel_b(fig.add_subplot(gs[0, 1]), D["stats"], D["duplicate_freq"])
draw_panel_c(fig.add_subplot(gs[1, 0]), D["bqr"])
draw_panel_d(fig, gs[1, 1], D["ms_table"])

stem = EXPORT_DIR / f"redux_{SAMPLE_ID}_{TUMOR_NAME}"
for ext in ("pdf", "svg", "png"):
    fig.savefig(f"{stem}.{ext}", bbox_inches="tight")
print("Wrote:", *(f"{stem}.{e}" for e in ("pdf", "svg", "png")))
plt.show()